## 한국어 뉴스 주제 분류 파인튜닝

기본 모델: `klue/bert-base`  
데이터셋: `klue/ynat`  
결과: 한국어 뉴스 제목을 주제별로 분류

파인튜닝 전에는 기본 모델이 뉴스 주제 분류용으로 학습되어 있지 않기 때문에 원하는 라벨을 제대로 예측하지 못한다.  
파인튜닝 후에는 한국어 뉴스 제목을 입력했을 때 정치, 경제, 사회, 생활문화, 세계, IT과학, 스포츠 중 하나로 분류할 수 있다.

영화리뷰 감정분석
기본 모델: beomi/kcbert-base
데이터셋: NSMC
분류 라벨: 부정, 긍정
결과: 한국어 영화 리뷰의 감정을 분류

## 1. 라이브러리 불러오기 및 환경 확인

In [ ]:
!pip install -q -U "datasets<4.0.0" transformers evaluate accelerate huggingface_hub

In [ ]:
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
import evaluate

print("GPU 사용 가능 여부:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))

GPU 사용 가능 여부: True
GPU 이름: Tesla T4


## 2. 데이터 불러오기

In [ ]:
from datasets import load_dataset

# 허깅페이스 정책 변경에 맞춰 klue/klue 풀네임과 trust_remote_code=True 적용
dataset = load_dataset("klue/klue", "ynat", trust_remote_code=True)

# 확신도를 높이기 위해 자르지 않고 전체 데이터를 모두 사용 (약 4.5만 개)
train_dataset = dataset["train"]
test_dataset = dataset["validation"]

print("학습 데이터 개수:", len(train_dataset))
print("검증 데이터 개수:", len(test_dataset))

학습 데이터 개수: 45678
검증 데이터 개수: 9107


## 3. 기본 모델과 토크나이저 불러오기

In [ ]:
model_name = "klue/bert-base"

# 토크나이저 불러오기
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 분류용 모델 불러오기 (뉴스 주제는 7개니까 num_labels=7 필수!)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=7
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 4. 문장을 토큰으로 변환하기


In [ ]:
def tokenize_function(examples):
    # 뉴스는 기사 제목인 'title' 컬럼을 사용
    return tokenizer(
        examples["title"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/45678 [00:00<?, ? examples/s]

Map:   0%|          | 0/9107 [00:00<?, ? examples/s]

## 5. 학습에 필요한 컬럼만 남기기


In [ ]:
# 모델이 학습할 때 숫자(토큰)와 정답(label)만 보도록 나머지 글자들은 다 지워주기
keep_cols = ["label", "input_ids", "token_type_ids", "attention_mask"]

remove_cols_train = [col for col in tokenized_train.column_names if col not in keep_cols]
remove_cols_test = [col for col in tokenized_test.column_names if col not in keep_cols]

tokenized_train = tokenized_train.remove_columns(remove_cols_train)
tokenized_test = tokenized_test.remove_columns(remove_cols_test)

## 6. 평가 지표 및 학습 설정하기

In [ ]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

# 3번 복습
training_args = TrainingArguments(
    output_dir="./ynat_klue_bert_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

## 7. 모델 파인튜닝 및 저장하기

In [ ]:
# 파인튜닝 시작
print("학습 시작합니다! 🚀")
trainer.train()

# 검증 데이터로 정확도 평가
trainer.evaluate()

# 파인튜닝된 모델과 토크나이저 저장
trainer.save_model("./my_korean_news_model")
tokenizer.save_pretrained("./my_korean_news_model")
print("모델 저장 완료됐습니다! 💾")

학습 시작합니다! 🚀


Epoch,Training Loss,Validation Loss,Accuracy
1,0.336523,0.395037,0.860217
2,0.303535,0.379321,0.871198
3,0.175277,0.436192,0.871967


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy
0.175277,0.379321,3,0.871198


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

모델 저장 완료됐습니다! 💾


## 8. 파인튜닝 후 실제 뉴스로 예측해보기

In [ ]:
after_classifier = pipeline(
    "text-classification",
    model="./my_korean_news_model",
    tokenizer="./my_korean_news_model",
    device=0 if torch.cuda.is_available() else -1
)

label_map = {
    "LABEL_0": "IT과학",
    "LABEL_1": "경제",
    "LABEL_2": "사회",
    "LABEL_3": "생활문화",
    "LABEL_4": "세계",
    "LABEL_5": "스포츠",
    "LABEL_6": "정치"
}

def predict_news_topic(text):
    result = after_classifier(text)[0]
    label = label_map[result["label"]]
    score = result["score"]

    print("입력 뉴스 제목:", text)
    print("예측 주제:", label)
    print("확신도:", round(score, 4))
    print("-" * 30)

my_texts = [
    "한은, 기준금리 3.5%로 동결... 가계부채 부담 고려",
    "한국 축구대표팀, 월드컵 32강 진출 실패...",
    "올여름 역대급 폭염 예고... 에어컨 판매량 급증",
    "새로운 AI 칩셋 발표... 기존 대비 성능 2배 향상"
]

print("📰 [실전 뉴스 제목 분류 테스트]")
for text in my_texts:
    predict_news_topic(text)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

📰 [실전 뉴스 제목 분류 테스트]
입력 뉴스 제목: 한은, 기준금리 3.5%로 동결... 가계부채 부담 고려
예측 주제: 경제
확신도: 0.9953
------------------------------
입력 뉴스 제목: 한국 축구대표팀, 월드컵 32강 진출 실패...
예측 주제: 스포츠
확신도: 0.9971
------------------------------
입력 뉴스 제목: 올여름 역대급 폭염 예고... 에어컨 판매량 급증
예측 주제: 생활문화
확신도: 0.7696
------------------------------
입력 뉴스 제목: 새로운 AI 칩셋 발표... 기존 대비 성능 2배 향상
예측 주제: IT과학
확신도: 0.9924
------------------------------
